In [ ]:

import cv2
import numpy as np
import pyautogui

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Access Webcam")

SCREEN_WIDTH, SCREEN_HEIGHT = pyautogui.size()

pyautogui.FAILSAFE = True

SMOOTHING = 5

previous_x = 0
previous_y = 0

FRAME_MARGIN = 100

while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    height, width = frame.shape[:2]

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60])
    upper_skin = np.array([25, 180, 255])

    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.rectangle(
        frame,
        (FRAME_MARGIN, FRAME_MARGIN),
        (width - FRAME_MARGIN, height - FRAME_MARGIN),
        (255, 0, 0),
        2
    )

    if contours:

        hand = max(contours, key=cv2.contourArea)

        if cv2.contourArea(hand) > 4000:

            fingertip = tuple(
                hand[hand[:, :, 1].argmin()][0]
            )

            finger_x, finger_y = fingertip

            if (
                FRAME_MARGIN < finger_x < width - FRAME_MARGIN
                and FRAME_MARGIN < finger_y < height - FRAME_MARGIN
            ):

                target_x = np.interp(
                    finger_x,
                    [FRAME_MARGIN, width - FRAME_MARGIN],
                    [0, SCREEN_WIDTH]
                )

                target_y = np.interp(
                    finger_y,
                    [FRAME_MARGIN, height - FRAME_MARGIN],
                    [0, SCREEN_HEIGHT]
                )

                current_x = previous_x + (
                    target_x - previous_x
                ) / SMOOTHING

                current_y = previous_y + (
                    target_y - previous_y
                ) / SMOOTHING

                pyautogui.moveTo(
                    int(current_x),
                    int(current_y)
                )

                previous_x = current_x
                previous_y = current_y

                cv2.circle(
                    frame,
                    fingertip,
                    12,
                    (0, 255, 0),
                    -1
                )

                cv2.drawContours(
                    frame,
                    [hand],
                    -1,
                    (0, 255, 0),
                    2
                )

    cv2.imshow("Virtual Mouse", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()